In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import pickle

In [ ]:
# 1. Load the Data
# Use the dataset file present in the workspace: 'car.csv'
df = pd.read_csv('car.csv')

In [ ]:
# 2. Data Preprocessing & Feature Engineering
# Derive 'Years_Since_Manufacture' from the 'Year' column
current_year = 2024
if 'Year' in df.columns:
    df['Years_Since_Manufacture'] = current_year - df['Year']
    df.drop('Year', axis=1, inplace=True)

# Drop irrelevant columns if they exist
for col in ['Car_Name']:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

# Encode categorical variables using one-hot encoding
df = pd.get_dummies(df, drop_first=True)

In [ ]:

# 3. Data Splitting
# Separate the target variable (Selling_Price) from the features
if 'Selling_Price' not in df.columns:
    raise ValueError('Expected target column "Selling_Price" not found in the dataset')
X = df.drop('Selling_Price', axis=1)
y = df['Selling_Price']

# Split the dataset into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 4. Model Selection & Hyperparameter Tuning
# Initialize the Random Forest Regressor
rf = RandomForestRegressor(random_state=42)

# Define a reasonable hyperparameter grid for RandomizedSearchCV
param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ['auto', 'sqrt'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 6],
}

# Optimize the hyperparameters using RandomizedSearchCV
rf_random = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=param_grid, 
    scoring='neg_mean_squared_error', 
    n_iter=10, 
    cv=5, 
    verbose=2, 
    random_state=42, 
    n_jobs=-1,
)

print("Training the model and tuning hyperparameters. This might take a minute...")
rf_random.fit(X_train, y_train)

In [ ]:
# 5. Model Evaluation
# Predict on the test set using the best estimator found
best_rf_model = rf_random.best_estimator_
predictions = best_rf_model.predict(X_test)

# Calculate and print the Mean Squared Error (MSE) and Root Mean Squared Error (RMSE)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
print(f"Model Mean Squared Error (MSE): {mse:.2f}")
print(f"Model Root Mean Squared Error (RMSE): {rmse:.2f}")

In [4]:
# 6. Deployment Preparation (Export the Model)
# Save the trained model to a pickle file so the Flask app can load it later
with open('random_forest_regression_model.pkl', 'wb') as file:
    pickle.dump(best_rf_model, file)

print("Model successfully saved as 'random_forest_regression_model.pkl'!")


Model successfully saved as 'random_forest_regression_model.pkl'!
